In [2]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from supabase import create_client
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env")

supabase = create_client(
    os.getenv("SUPABASE_URL"),
    os.getenv("SUPABASE_KEY")
)

result = supabase.table("watches").select(
    "watched_date, rating, rewatch, tags, films(name, year, genres, runtime_mins, original_lang)"
).execute()

df = pd.json_normalize(result.data)
df.columns = [c.replace("films.", "") for c in df.columns]
df["watched_date"] = pd.to_datetime(df["watched_date"])
df["year"] = df["year"].astype("Int64")
df["month"] = df["watched_date"].dt.month
df["year_watched"] = df["watched_date"].dt.year

# exploding time (just make every genre pair get its own row)
df_genres = df.explode("genres").dropna(subset=["genres"])

print(df.shape)
print(df.columns.tolist())
print(df.head(3))

(586, 11)
['watched_date', 'rating', 'rewatch', 'tags', 'name', 'year', 'genres', 'runtime_mins', 'original_lang', 'month', 'year_watched']
  watched_date  rating  rewatch  tags  \
0   2021-02-15     5.0    False  None   
1   2022-07-28     4.0    False  None   
2   2022-07-29     4.0    False  None   

                                                name  year  \
0                                  Chungking Express  1994   
1                                        Fire Island  2022   
2  The French Dispatch of the Liberty, Kansas Eve...  2021   

                     genres  runtime_mins original_lang  month  year_watched  
0  [Drama, Comedy, Romance]           103            cn      2          2021  
1         [Comedy, Romance]           105            en      7          2022  
2           [Drama, Comedy]           108            en      7          2022  


In [3]:
#genre distribution
genre_counts = df_genres.groupby("genres").size().sort_values(ascending=False)
print("Top 20 genres watched:")
print(genre_counts.head(20))
print()

#weighted by rating
genre_ratings = df_genres.groupby("genres")["rating"].agg(["mean", "count"]).sort_values("count", ascending=False)
genre_ratings.columns = ["avg_rating", "count"]
print("Genre ratings (min 10 watches):")
print(genre_ratings[genre_ratings["count"] >= 10].sort_values("avg_rating", ascending=False).to_string())

Top 20 genres watched:
genres
Drama              299
Comedy             215
Romance            165
Thriller           121
Action             117
Horror              85
Adventure           80
Crime               68
Science Fiction     65
Family              49
Mystery             48
Animation           46
Fantasy             39
Music               17
History             13
Documentary         12
TV Movie             3
War                  2
Western              1
dtype: int64

Genre ratings (min 10 watches):
                 avg_rating  count
genres                            
History            4.230769     13
Documentary        4.166667     12
Music              4.058824     17
Drama              3.946488    299
Mystery            3.802083     48
Crime              3.727941     68
Romance            3.700000    165
Animation          3.695652     46
Thriller           3.582645    121
Horror             3.547059     85
Comedy             3.451163    215
Family             3.428571     

In [4]:
# eras
bins = [1900, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2030]
labels = ["pre-1950", "1950s", "1960s", "1970s", "1980s", "1990s", "2000s", "2010s", "2015-2019", "2020s"]

df["era"] = pd.cut(df["year"], bins=bins, labels=labels, right=False)

era_stats = df.groupby("era", observed=True)["rating"].agg(["mean", "count"])
era_stats.columns = ["avg_rating", "count"]

print("Watches and ratings by era:")
print(era_stats.to_string())

Watches and ratings by era:
           avg_rating  count
era                         
pre-1950     3.833333      3
1950s        4.500000      2
1960s        4.045455     11
1970s        4.314815     27
1980s        4.064516     31
1990s        4.077381     84
2000s        3.780769    130
2010s        3.564286     70
2015-2019    3.294118     68
2020s        3.380503    159


In [ ]:
# note to self: bias recommender towards older films if other matches are equal

In [5]:
lang_stats = df.groupby("original_lang")["rating"].agg(["mean", "count"])
lang_stats.columns = ["avg_rating", "count"]
lang_stats = lang_stats[lang_stats["count"] >= 5].sort_values("avg_rating", ascending=False)

lang_map = {
    "en": "English", "fr": "French", "ja": "Japanese",
    "ko": "Korean", "it": "Italian", "es": "Spanish",
    "de": "German", "zh": "Mandarin", "cn": "Cantonese",
    "pt": "Portuguese", "da": "Danish", "sv": "Swedish",
    "pl": "Polish", "hi": "Hindi", "fa": "Persian"
}

lang_stats.index = lang_stats.index.map(lambda x: lang_map.get(x, x))
print("Ratings by language (min 5 watches):")
print(lang_stats.sort_values("avg_rating", ascending=False).to_string())

Ratings by language (min 5 watches):
               avg_rating  count
original_lang                   
Cantonese        5.000000      5
Mandarin         4.400000      5
Japanese         4.388889     27
French           4.333333     15
Korean           4.250000      6
Italian          4.100000      5
Hindi            3.747573    103
English          3.516291    399


In [6]:
# pretty sure all the bollywood movies are me showing ria and irene a brother nefarious or two

hindi_social = df[
    (df["original_lang"] == "hi") & 
    (df["tags"].apply(lambda x: "social" in x if isinstance(x, list) else False))
]
hindi_solo = df[
    (df["original_lang"] == "hi") & 
    (df["tags"].apply(lambda x: "social" not in x if isinstance(x, list) else True))
]

print("Hindi social watches:", len(hindi_social), "avg rating:", round(hindi_social["rating"].mean(), 2))
print("Hindi solo watches:", len(hindi_solo), "avg rating:", round(hindi_solo["rating"].mean(), 2))

Hindi social watches: 67 avg rating: 3.68
Hindi solo watches: 36 avg rating: 3.88
